# IMPORT

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import make_scorer, mean_squared_log_error

# Load Dataset

In [3]:
TRAIN_PATH = "train.csv"
TEST_PATH  = "test.csv"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

TARGET = "Rings"
ID_COL = "id"

# Basic checks
assert TARGET in train.columns, f"Target column '{TARGET}' not found in train.csv"
assert ID_COL in train.columns and ID_COL in test.columns, f"ID column '{ID_COL}' must exist in both train and test"

X = train.drop(columns=[TARGET])
y = train[TARGET].copy()

# RMSLE scorer (competition metric)

In [4]:
def rmsle(y_true, y_pred):
    # RMSLE requires non-negative predictions
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

rmsle_scorer = make_scorer(rmsle, greater_is_better=False) 

# Preprocessing (numeric + categorical)

In [5]:
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols and c != ID_COL]  # exclude id from modeling

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

# MODEL 1: Ridge Regression (linear baseline) + log1p target

In [6]:
ridge = Ridge(alpha=5.0, random_state=0)

model1 = Pipeline(steps=[
    ("preprocess", preprocess),
    ("reg", TransformedTargetRegressor(
        regressor=ridge,
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

# MODEL 2: HistGradientBoostingRegressor (nonlinear) + log1p

In [7]:
hgb = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=6,
    max_iter=600,
    random_state=0
)

model2 = Pipeline(steps=[
    ("preprocess", preprocess),
    ("reg", TransformedTargetRegressor(
        regressor=hgb,
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

# Cross-validation evaluation

In [8]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)

m1_scores = cross_val_score(model1, X.drop(columns=[ID_COL]), y, cv=cv, scoring=rmsle_scorer)
m2_scores = cross_val_score(model2, X.drop(columns=[ID_COL]), y, cv=cv, scoring=rmsle_scorer)

print("MODEL 1 (Ridge) RMSLE CV mean:", -m1_scores.mean(), " | std:", m1_scores.std())
print("MODEL 2 (HGB)   RMSLE CV mean:", -m2_scores.mean(), " | std:", m2_scores.std())

MODEL 1 (Ridge) RMSLE CV mean: 0.16450322484703744  | std: 0.0007049208225400694
MODEL 2 (HGB)   RMSLE CV mean: 0.14971064839379805  | std: 0.0007628815390009466


# Fit models on full training

In [9]:
model1.fit(X.drop(columns=[ID_COL]), y)
model2.fit(X.drop(columns=[ID_COL]), y)

,steps,"[('preprocess', ...), ('reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Predict on test + create submissions (id, Rings)

In [10]:
test_features = test.drop(columns=[ID_COL])

pred1 = np.clip(model1.predict(test_features), 0, None)
pred2 = np.clip(model2.predict(test_features), 0, None)

sub1 = pd.DataFrame({ID_COL: test[ID_COL], TARGET: pred1})
sub2 = pd.DataFrame({ID_COL: test[ID_COL], TARGET: pred2})

sub1.to_csv("submission_ridge.csv", index=False)
sub2.to_csv("submission_hgb.csv", index=False)

print("\nSaved submissions:")
print(" - submission_ridge.csv")
print(" - submission_hgb.csv")


Saved submissions:
 - submission_ridge.csv
 - submission_hgb.csv
